# 🧬 CancerNet Training on GPU

This notebook trains CancerNet on a **free T4 GPU** in Google Colab.

**Before running:**
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Have your `cancer-detection.zip` ready (your project folder zipped)
3. Have your `kaggle.json` API key ready (if downloading dataset from Kaggle)

---

## Step 1: Verify GPU is available

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

!nvidia-smi

## Step 2: Upload your project

**On your laptop**, zip your project folder (WITHOUT the `data/raw` and `venv` folders to keep the zip small):

You can do this manually or run this in PowerShell:
```powershell
cd C:\Users\Amal Raju\Desktop
Compress-Archive -Path cancer-detection\configs, cancer-detection\src, cancer-detection\scripts, cancer-detection\notebooks, cancer-detection\tests, cancer-detection\requirements.txt -DestinationPath cancer-detection-code.zip
```

Then upload the zip below:

In [ ]:
from google.colab import files
print("Upload your cancer-detection-code.zip file:")
uploaded = files.upload()

In [ ]:
# Unzip the project
import os
!rm -rf /content/cancer-detection
!unzip -q cancer-detection-code.zip -d /content/cancer-detection

# Check if files are nested inside an extra folder
if os.path.exists('/content/cancer-detection/cancer-detection'):
    !mv /content/cancer-detection/cancer-detection/* /content/cancer-detection/
    !rm -rf /content/cancer-detection/cancer-detection

# Create necessary directories
!mkdir -p /content/cancer-detection/data/raw
!mkdir -p /content/cancer-detection/data/processed
!mkdir -p /content/cancer-detection/data/splits
!mkdir -p /content/cancer-detection/checkpoints
!mkdir -p /content/cancer-detection/logs
!mkdir -p /content/cancer-detection/results

%cd /content/cancer-detection
!ls -la

## Step 3: Install dependencies

In [ ]:
# Install PyTorch Geometric and its dependencies
!pip install -q torch-geometric
!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html

# Install remaining dependencies
!pip install -q timm albumentations opencv-python scikit-image scikit-learn \
    omegaconf tqdm grad-cam networkx scipy Pillow

## Step 4: Download dataset from Kaggle

This downloads the **Breast Histopathology Images (IDC)** dataset.

**To get your `kaggle.json`:**
1. Go to [kaggle.com/settings](https://www.kaggle.com/settings)
2. Scroll to **API** section
3. Click **Create New Token** → downloads `kaggle.json`

In [ ]:
# Upload Kaggle API key
from google.colab import files
print("Upload your kaggle.json:")
files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Download the IDC breast cancer dataset
# If your dataset has a different name, change the command below
!pip install -q kaggle
!kaggle datasets download -d paultimothymooney/breast-histopathology-images -p /content/dataset --unzip

# Move patient folders to data/raw/
import shutil, glob

# The dataset may be nested — find the patient folders (numbered directories)
!ls /content/dataset/

In [ ]:
# Copy patient folders to data/raw (adjust path if nested differently)
import os
import shutil

# Find where the numbered patient folders are
dataset_root = '/content/dataset'
for root, dirs, files_list in os.walk(dataset_root):
    # Look for directories that are numbered (patient IDs)
    numbered_dirs = [d for d in dirs if d.isdigit()]
    if len(numbered_dirs) > 50:  # Found the patient folders
        print(f"Found {len(numbered_dirs)} patient folders in: {root}")
        for d in numbered_dirs:
            src = os.path.join(root, d)
            dst = os.path.join('/content/cancer-detection/data/raw', d)
            if not os.path.exists(dst):
                shutil.copytree(src, dst)
        break

# Also copy IDC_regular folder if it exists
for root, dirs, files_list in os.walk(dataset_root):
    for d in dirs:
        if d.startswith('IDC_regular'):
            src = os.path.join(root, d)
            dst = os.path.join('/content/cancer-detection/data/raw', d)
            if not os.path.exists(dst):
                shutil.copytree(src, dst)

print(f"\nTotal folders in data/raw: {len(os.listdir('/content/cancer-detection/data/raw'))}")

## Step 5: Quick sanity check

In [ ]:
%cd /content/cancer-detection

# Verify everything is in place
import os
print("Project files:")
for item in sorted(os.listdir('.')):
    if item not in ['venv', '__pycache__', '.git']:
        print(f"  {item}")

print(f"\nPatient folders in data/raw: {len(os.listdir('data/raw'))}")
print(f"\nConfig to use: configs/config_colab.yaml")

# Quick import test
from omegaconf import OmegaConf
cfg = OmegaConf.load('configs/config_colab.yaml')
print(f"\nConfig loaded successfully!")
print(f"  Device: {cfg.project.device}")
print(f"  Mixed precision: {cfg.training.mixed_precision}")
print(f"  Batch size: {cfg.training.batch_size}")
print(f"  Epochs: {cfg.training.epochs}")

## Step 6: 🚀 Start Training!

This will use the GPU-optimized config. Expected time: **~3-4 hours** on T4.

In [ ]:
%cd /content/cancer-detection
!python scripts/train.py --config configs/config_colab.yaml

## Step 7: Download trained model

After training completes, download your best checkpoint:

In [ ]:
from google.colab import files

# Download the best model
if os.path.exists('checkpoints/best_model.pth'):
    files.download('checkpoints/best_model.pth')
    print("✅ Best model downloaded!")
else:
    print("⚠️ No best_model.pth found. Check if training completed.")
    !ls checkpoints/

In [ ]:
# Download all checkpoints and logs as a zip
!zip -r training_results.zip checkpoints/ logs/ results/
files.download('training_results.zip')
print("✅ All results downloaded!")

---
## ⚠️ Important Notes

- **Colab disconnects after ~90 min of inactivity.** Keep the browser tab open!
- **Free tier has ~12 hour session limit.** Training should finish well within this.
- If training is interrupted, your checkpoints are saved — you can resume from the last epoch.
- **GPU memory issues?** Reduce `batch_size` from 16 to 8 in `config_colab.yaml`.